# U6-L2：动作录制

**课程目标**

1. 能说明拖动示教录制的原理——拖的时候记录每个关节角度随时间的变化。
2. 能按顺序说出录制经过的几步（进示教上下文 → 开始录 → 拖 → 停止录 → 保存），并说出轨迹文件的后缀。
3. 能测出录制时长，并读出保存后的轨迹数据（关节名、采样点数、时长）与录制时长核对。

## 一、本课要解决的问题

上节课把机器人获得动作的三条路线过了一遍：规则驱动、模仿学习、强化学习。本课程落在模仿学习的采集端——示教。上节课读的那条轨迹是教师录好的，我们只是「看」了它。

能不能自己录一条？人拖着机器人做一个动作，让机器人把这个动作记下来。

> **人拖着机器人做一遍动作，怎么把它变成一条可以保存、以后还能重演的轨迹数据？**

## 二、示教录制的原理

录制的原理和「录像」很像：录像记录的是画面随时间的变化，录制轨迹记录的是**每个关节角度随时间的变化**。

拖动的时候，机器人一边被你带着走，一边按固定间隔记下所有关节的角度；停下来的那一刻，这些角度按时间顺序连起来，就是一条轨迹。这就是上节课读到的 `.btraj` 文件里存的东西。

## 三、录制一条轨迹

下面自己录一条。录制过程是：进入示教状态 → 开始录 → 拖一个动作 → 停止录 → 保存。

⚠️ **安全**：本课需要让机器人进入<font color="red">行走模式</font>并动手拖动它。请在平整、无障碍的场地上进行，全程留人在伸手可及的位置；拖动时动作轻缓，不要用力掰关节。出现意外可以直接通过按钮把机器人切到**准备模式**。

📌 **前置**：机器人已开机、配网。

**STEP 01：** 连接机器人

In [ ]:
from boosteros.robots.booster import BoosterRobot
from boosteros.types import TrajectoryData

robot = BoosterRobot()

print("已连接：", robot.robot_info.model, robot.robot_info.serial_number)
print("当前模式：", robot.get_mode())

### 3.1 录制接口

**开始下面程序前，机器人应处于行走模式。** 如果机器人当前不在行走模式：先用按钮把机器人切到准备模式，等它站直站稳；再切到行走模式。确认进入行走模式后，再运行下面的代码。

在 BoosterOS SDK 里，录制用到示教管理器的三个成员：

- `start_recording()`：开始录制，从这一刻起，机器人开始记关节角度。**必须在 `with` 上下文里调用**。
- `stop_recording()`：停止录制，返回一条轨迹数据（`TrajectoryData`）；如果没录到有效轨迹，返回 `None`。
- `get_recording_duration()`：返回当前录制持续的秒数，没在录时是 `0.0`。

**STEP 02：** 开始录制，拖一个动作

In [ ]:
with robot.hand_guiding_manager as hand_guiding:
    hand_guiding.start_recording()
    input("拖着机器人做一个动作（比如挥一下手），做完按回车停止录制。")
    traj = hand_guiding.stop_recording()

if traj is not None:
    print("录制完成，采样点数：", len(traj.points))
else:
    print("没有录到有效轨迹，请重新拖一次")

**STEP 03：** 保存轨迹

<font color="red">提示：</font>轨迹文件的后缀是 `.btraj`。如果上一步屏幕显示「没有录到有效轨迹」，先重新跑一次 STEP 02，再跑这一格。

In [ ]:
if traj is not None:
    traj.save("my_action.btraj")
    print("已保存到 my_action.btraj")
    print("轨迹时长：", traj.duration.seconds, "秒")
    print("采样点数：", len(traj.points))

**STEP 04：** 读回验证

把刚保存的轨迹再读出来，核对关节名、时长、采样点数，和上面保存时打出来的一致。

In [ ]:
traj_back = TrajectoryData.load("my_action.btraj")

print("关节名：", traj_back.joint_names)
print("时长：", traj_back.duration.seconds, "秒")
print("采样点数：", len(traj_back.points))

### 3.2 收尾

实验做完，机器人还停在行走模式。收尾顺序：先用按钮把机器人切回**准备模式**，等它站直稳住；再由人工扶着切到**阻尼模式**（damping），扶机器人坐下，确认坐稳。不要从行走模式直接切到阻尼模式（那样机器人会当场瘫倒）。

## 四、观察与记录

### 4.1 录制结果

| 项目 | 记录 |
| :--- | :--- |
| 录的动作 | 　　　　 |
| 关节名 | 　　　　 |
| 轨迹时长（秒） | 　　　　 |
| 采样点数 | 　　　　 |
| 保存的文件名 | 　　　　 |

### 4.2 检查点

- 是否说得出录制时机器人记下来的是什么（每个关节角度随时间的变化）。
- 是否能按顺序说出录制的几步，并说出轨迹文件的后缀。
- 是否能把保存的轨迹再读出来，并且读出来的时长、采样点数和保存时一致。

### 4.3 思考题

- 录制的时候拖得快一点、拖得慢一点，录出来的轨迹数据会有什么差别？
- 如果拖到一半停了几秒再继续拖，轨迹里会记下这段「停顿」吗？